# Heart Disease Dataset EDA

This notebook performs exploratory data analysis on `data/heart-disease.csv`.

It includes:
- Basic schema checks
- Missing value analysis
- Outlier detection for numeric features (IQR/boxplots)
- Feature-type inference (numeric vs categorical)
- Target distribution and correlations


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set(style="whitegrid")
%matplotlib inline

# Robust path resolution (works whether you launch Jupyter from repo root or notebook folder)
candidate_paths = [
    Path("data") / "heart-disease.csv",
    Path("..") / "data" / "heart-disease.csv",
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not locate data/heart-disease.csv")

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

df.describe(include="all")

In [ ]:
target_col_candidates = [c for c in df.columns if c.lower() in ["target", "label", "output", "y", "class"]]
target_col = "target" if "target" in df.columns else (target_col_candidates[0] if target_col_candidates else df.columns[-1])
feature_cols = [c for c in df.columns if c != target_col]

print("Using target column:", target_col)
print("Number of features:", len(feature_cols))

In [ ]:
# Missing value analysis
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(4)

missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
missing_report[missing_report.missing_count > 0]

In [ ]:
# Visual missingness heatmap (works best when missingness exists)
plt.figure(figsize=(12, 5))
sns.heatmap(df.isna(), cbar=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.show()

In [ ]:
# Target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df)
plt.title("Target Distribution")
plt.show()

print(df[target_col].value_counts(dropna=False))

In [ ]:
# Infer numeric vs categorical (data-driven)
categorical_cols = []
numeric_cols = []

for col in feature_cols:
    s = df[col]
    if s.dtype == "object" or str(s.dtype).lower().startswith("category"):
        categorical_cols.append(col)
    elif pd.api.types.is_integer_dtype(s.dtype):
        # Treat low-cardinality integer columns as categorical (e.g., sex/cp/restecg)
        if s.nunique(dropna=True) <= 10:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)
    else:
        numeric_cols.append(col)

print("Numeric cols:", numeric_cols)
print("Categorical cols:", categorical_cols)

In [ ]:
# Numeric distributions and outliers (boxplots)
if numeric_cols:
    for col in numeric_cols:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=df[col])
        plt.title(f"Boxplot: {col}")
        plt.show()

# IQR-based outlier count summary
outlier_summary = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    outlier_count = ((df[col] < lo) | (df[col] > hi)).sum()
    outlier_summary.append((col, int(outlier_count)))

pd.DataFrame(outlier_summary, columns=["column", "iqr_outlier_count"]).sort_values(
    "iqr_outlier_count", ascending=False
)

In [ ]:
# Categorical distributions
if categorical_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(categorical_cols) / n_cols))
    plt.figure(figsize=(15, 4 * n_rows))

    for i, col in enumerate(categorical_cols, start=1):
        plt.subplot(n_rows, n_cols, i)
        sns.countplot(x=col, data=df, color="#4C78A8")
        plt.title(col)
        plt.tight_layout()

        plt.show()


In [ ]:
# Correlation heatmap for numeric features (if any)
if numeric_cols:
    corr = df[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Correlation Heatmap (Numeric Features)")
    plt.show()
else:
    print("No numeric columns inferred for correlation heatmap.")

## Notes for Modeling

Based on this EDA, the training pipeline:
- Imputes missing values (`median` for numeric, `most_frequent` for categorical)
- Caps numeric outliers using the IQR rule
- One-hot encodes categorical features
- Applies `StandardScaler` to numeric features
- Trains an `XGBoost` classifier with cross-validated hyperparameter tuning
